# Train yawn frame classifier (face ROI architecture)

Notebook này huấn luyện model phân loại **face frame**, không huấn luyện detector box miệng.

Pipeline runtime tương ứng:

```text
Camera frame → face ROI → classifier → no_yawn / yawn
                                      ↓
                           temporal sliding-window decision
```

Dữ liệu ảnh tĩnh không có thứ tự video, vì vậy notebook này huấn luyện frame baseline. Nó không giả nhận rằng đã triển khai YawnNet/Transformer sequence model. Lớp smoothing sẽ được dùng ở runtime, sau khi có xác suất `yawn` của từng frame.

## 1. Cài đặt và tham số

In [ ]:
from pathlib import Path

CLASSES = ('no_yawn', 'yawn')
SOURCE = None  # Có thể sửa cell discovery nếu cần.
PREPARED = Path('/kaggle/working/yawn_classification')
RUNS = Path('/kaggle/working/runs_yawn')
RUN_NAME = 'yawn_frame_classifier'


## 2. Tìm và kiểm tra dataset classification

In [ ]:
from pathlib import Path
from collections import Counter
import json
import random
import shutil
import hashlib
from PIL import Image

SEED = 42
random.seed(SEED)

# Để None để tự tìm trong /kaggle/input. Có thể đặt đường dẫn thủ công nếu Kaggle
# dùng tên mount khác, ví dụ: Path('/kaggle/input/ten-dataset/dataset_new').
# Giữ None để tự tìm; nếu cần, sửa dòng này thành Path('...').
SOURCE = None
EXPECTED = ('train', 'test')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def has_classification_layout(root: Path, classes):
    return all(
        (root / split).is_dir()
        and all((root / split / cls).is_dir() for cls in classes)
        for split in EXPECTED
    )


def discover_source(classes):
    if SOURCE is not None:
        root = Path(SOURCE)
        if not has_classification_layout(root, classes):
            raise FileNotFoundError(
                f'SOURCE không có cấu trúc train/test/{classes}: {root}'
            )
        return root

    search_roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    candidates = []
    for base in search_roots:
        if not base.exists():
            continue
        for train_dir in base.rglob('train'):
            root = train_dir.parent
            if has_classification_layout(root, classes):
                candidates.append(root)
    # Loại trùng và ưu tiên đường dẫn nông hơn.
    candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
    if not candidates:
        raise FileNotFoundError(
            f'Không tìm thấy dataset classification có train/test và classes={classes}. '
            'Hãy đặt SOURCE thủ công ở cell này.'
        )
    print('Các dataset tìm thấy:')
    for p in candidates:
        print(' -', p)
    return candidates[0]


def image_files(folder):
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def inventory(root, classes):
    counts = {}
    for split in EXPECTED:
        counts[split] = {cls: len(image_files(root / split / cls)) for cls in classes}
    return counts


In [ ]:
YAWN_SOURCE = discover_source(CLASSES)
print('SOURCE =', YAWN_SOURCE)
print(json.dumps(inventory(YAWN_SOURCE, CLASSES), indent=2))


## 3. Tạo train/validation/test

In [ ]:
def prepare_classification_dataset(source, destination, classes, val_fraction=0.20):
    """Create train/val/test folders from train/test class folders.

    The source has no validation split, so val is a stratified split of source/train.
    Test is copied without being used for model-selection thresholds.
    """
    if destination.exists():
        shutil.rmtree(destination)
    for split in ('train', 'val', 'test'):
        for cls in classes:
            (destination / split / cls).mkdir(parents=True, exist_ok=True)

    manifest = []
    for cls in classes:
        train_items = image_files(source / 'train' / cls)
        test_items = image_files(source / 'test' / cls)
        random.Random(SEED).shuffle(train_items)
        n_val = max(1, round(len(train_items) * val_fraction))
        groups = {
            'val': train_items[:n_val],
            'train': train_items[n_val:],
            'test': test_items,
        }
        for split, items in groups.items():
            for index, src in enumerate(items):
                # Prefix avoids collisions when different class folders reuse filenames.
                dst_name = f'{cls}_{index:06d}_{src.name}'
                dst = destination / split / cls / dst_name
                shutil.copy2(src, dst)
                manifest.append({'split': split, 'class': cls, 'source': str(src), 'copied': str(dst)})

    (destination / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return manifest


def verify_dataset(root, classes):
    report = {}
    for split in ('train', 'val', 'test'):
        report[split] = {}
        for cls in classes:
            files = image_files(root / split / cls)
            good = 0
            for path in files:
                try:
                    with Image.open(path) as im:
                        im.verify()
                    good += 1
                except Exception as exc:
                    raise RuntimeError(f'Ảnh lỗi: {path}: {exc}')
            report[split][cls] = good
    return report


In [ ]:
manifest = prepare_classification_dataset(YAWN_SOURCE, PREPARED, CLASSES, val_fraction=0.20)
print(json.dumps(verify_dataset(PREPARED, CLASSES), indent=2))
print('Samples copied:', len(manifest))


## 4. Train yawn frame classifier

Model nhận face crop/frame 224×224 và trả về xác suất cho `no_yawn` hoặc `yawn`. Không dùng toàn-frame detection box.

In [ ]:
# Cài dependency trong Kaggle nếu cần.
%pip install -q ultralytics

from ultralytics import YOLO
import torch

EPOCHS = 40
IMG_SIZE = 224
BATCH = 32
WORKERS = 2
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO('yolov8n-cls.pt')
model.train(
    data=str(PREPARED),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    seed=SEED,
    project=str(RUNS),
    name=RUN_NAME,
    patience=10,
    device=DEVICE,
    workers=WORKERS,
    pretrained=True,
)

BEST = RUNS / RUN_NAME / 'weights' / 'best.pt'
print('Best checkpoint:', BEST)


## 5. Đánh giá và lưu checkpoint

In [ ]:
# Đánh giá trên test một lần sau khi chốt hyperparameters.
best_model = YOLO(str(BEST))
test_metrics = best_model.val(
    data=str(PREPARED),
    split='test',
    imgsz=IMG_SIZE,
    project=str(RUNS),
    name=f'{RUN_NAME}_test',
    device=DEVICE,
)
print(test_metrics)

# In class mapping để runtime không bị đảo nhãn.
print('Class names:', best_model.names)
print('Checkpoint dùng cho camera:', BEST)


## 6. Temporal decision dùng sau khi train

Do dataset hiện tại không có clip và thứ tự frame, không thể train sequence model đúng nghĩa. Runtime nên đưa xác suất yawn qua cửa sổ thời gian để giảm nhảy nhãn một frame.

In [ ]:
from collections import deque

class TemporalYawnDecision:
    def __init__(self, window=16, min_yawn_frames=8,
                 enter_threshold=0.70, exit_threshold=0.35):
        self.probs = deque(maxlen=window)
        self.min_yawn_frames = min_yawn_frames
        self.enter_threshold = enter_threshold
        self.exit_threshold = exit_threshold
        self.state = 'no_yawn'

    def update(self, yawn_probability):
        self.probs.append(float(yawn_probability))
        if len(self.probs) < self.probs.maxlen:
            return self.state
        count = sum(p >= self.enter_threshold for p in self.probs)
        mean = sum(self.probs) / len(self.probs)
        if (self.state == 'no_yawn'
                and count >= self.min_yawn_frames
                and mean >= self.enter_threshold):
            self.state = 'yawn'
        elif self.state == 'yawn' and mean <= self.exit_threshold:
            self.state = 'no_yawn'
        return self.state

# Runtime gọi một lần cho mỗi face frame đã xử lý:
# state = decision.update(probability_of_yawn)


## 7. Hợp đồng input cho camera runtime

Runtime phải detect/locate một khuôn mặt, crop face ROI, resize về kích thước model và đọc xác suất class. Không gọi `.plot()` và không mong checkpoint này trả về box miệng.

Muốn chuyển sang YawnNet-style sequence model sau này, cần thu video clip có thứ tự frame, giữ clip-disjoint train/val/test, rồi mới tạo sequence 64→16 frame.